### Env and LLM initialisation

In [ ]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
import os

load = load_dotenv('./../.env', override=True)


# ollama_cloud_llm = ChatOllama(
#     base_url="http://localhost:11434/",  # Ollama cloud endpoint
#     model="devstral-small-2:24b-cloud", #gemini-3-flash-preview:cloud #qwen3.5:cloud
#     temperature=0.5,
#     max_tokens=1000,
#     headers={
#         "Authorization": f"Bearer {os.getenv('OLLAMA_CLOUD_API_KEY')}"  # Cloud auth
#     }
# )

ollama_local_llm = ChatOllama(
    base_url="http://localhost:11434/",
    model="gemma4:latest",
    temperature=0.5,
    max_tokens=500,
    num_gpu=999
)

### Bringing back the code from pervious section for tool binding with LLM

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

wikipidia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

search_tool = DuckDuckGoSearchRun()

@tool
def add_numbers(a: int, b:int) -> int:
    "Add two number and return results."
    return  int(a) + int(b)

@tool
def subtract_numbers(a: int, b:int) -> int:
    "Subtract two number and return results."
    return  int(a) - int(b)

@tool
def multiply_numbers(a: int, b:int) -> int:
    "Multiply two number and return results."
    return  int(a) * int(b)

tools = [wikipidia, add_numbers, subtract_numbers, multiply_numbers]

print(tools)


### Agent code

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    tools=tools,
    model=ollama_local_llm,
    system_prompt="You are a helpful assistant that can answer questions using the available tools."
)

query = "What is the sum of 2 and 4? Also, Did donald trump won the 2024 presidential election and became president in 2025?"

result = agent.invoke ({"messages": [HumanMessage(content=query)]})

print(result)
print(result["messages"][-1].content)

### Agent code with ChatPromptTemplate

In [ ]:
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

agent = create_agent(
    tools=tools,
    model=ollama_local_llm,
    system_prompt="You are a helpful assistant who is actually expert in Maths and latest news. You can answer questions using the available tools."
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant who is actually expert in Maths and latest news. You can answer questions using the available tools."),
    ("user", "What is the sum of 2 and 3?"),
    ("user", "What is latest movie of Tom Cruise hitting theatter in 2025?"),
    ("user", "Give me both the ansewers in JSON format")
])

result = agent.invoke({"messages": prompt_template.format_messages()})

print(result)
print(result["messages"][-1].content)

### Using Playwright Browser Toolkit

In [ ]:
#pip install -qU  playwright
#pip install -qU  lxml

In [ ]:
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import (
    create_async_playwright_browser,
)
import nest_asyncio

nest_asyncio.apply()



### Instantiating a browser toolkit

In [ ]:
async_browser = create_async_playwright_browser()
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools()
tools


In [ ]:
tools_by_name = {tool.name: tool for tool in tools}
navigate_tool = tools_by_name["navigate_browser"]
get_element_tool = tools_by_name["get_elements"]
navigate_tool, get_element_tool

In [ ]:
navigate_tool.arun({"url": "http://eaapp.somee.com/Employee/"})


In [ ]:
get_element_tool.arun({
    "selector": "td",
    "action": "innerText"
})


### Agent with Playwright Browser Toolkit

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    tools=tools,
    model=ollama_local_llm,
    system_prompt="You are a helpful assistant that can answer questions using the available tools."
)

query = "What are the links in http://eaapp.somee.com/Employee/ page?"

result = await agent.ainvoke({"messages": [HumanMessage(content=query)]})

print(result["messages"][-1].content)